# Sesión 10 — Validación cruzada: ¿es bueno de verdad o tuve suerte con la partición?

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 4 — Evaluación y optimización

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Explicar por qué **un solo** `train_test_split` puede engañar, sobre todo con pocos datos.
2. Usar **validación cruzada** (`cross_val_score`, `StratifiedKFold`) para estimar el desempeño con promedio y dispersión.
3. **Comparar modelos** con validación cruzada y decidir si una diferencia es real.
4. Construir un **`Pipeline`** para que todo lo que aprende de los datos (escalar, balancear…) se ajuste dentro de cada ronda, y reconocer la fuga cuando no se hace.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                                    ▲▲▲▲▲▲▲▲▲▲▲▲
   HOY: medir bien. Un número con su incertidumbre, y sin que el modelo haya visto el examen.
```

🔍 **Predice** · ✏️ **Completa** (andamiaje **bajo**) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Datos:** Bank Marketing otra vez (Sesiones 5 y 6): 45 211 llamadas, 11,7 % "sí", sin `duration`. Así podemos comparar con lo que ya sabemos.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"bank.csv": "datos/bank.csv", "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo.
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("evaluación", modulo=4, subtitulo="HOY: medir bien — un número con su incertidumbre, sin ver el examen");

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

bank = pd.read_csv("bank.csv")
y = (bank["deposit"] == "yes").astype(int)
X = pd.get_dummies(bank.drop(columns=["deposit", "duration"]), drop_first=True, dtype=int)
print(X.shape, f"· 'sí' = {y.mean():.1%}")

## 1. La lotería del split

Hasta ahora siempre hicimos **un** `train_test_split` con `random_state=42`. ¿Qué pasa si cambiamos solo esa semilla? Un árbol de profundidad 5, 30 particiones distintas:

### 🔍 Predice
Con los 45 211 clientes, ¿cuánto cree que varía el AUC de prueba entre las 30 particiones? ¿Y si tuviéramos solo **1 500** clientes (el tamaño típico de un proyecto del curso)?

**Tu predicción:** con 45 211: de ____ a ____ · con 1 500: de ____ a ____

In [ ]:
def auc_30_splits(X, y):
    scores = []
    for semilla in range(30):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=semilla, stratify=y)
        arbol = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tr, y_tr)
        scores.append(roc_auc_score(y_te, arbol.predict_proba(X_te)[:, 1]))
    return scores

pocos = bank.sample(1500, random_state=0).index
fg.loteria_split({"45 211 clientes": auc_30_splits(X, y), "1 500 clientes": auc_30_splits(X.loc[pocos], y.loc[pocos])});

## 2. Validación cruzada: muchos exámenes en vez de uno

Primero apartamos una **prueba** (20 %) que no vamos a tocar hasta el final. Con el 80 % restante hacemos **validación cruzada**: lo partimos en *k* pedazos (*folds*), entrenamos *k* veces, cada vez dejando un fold distinto para validar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
fg.cv_folds(k=5);

### 📐 Fundamento
Con $k$ folds se obtienen $k$ scores $s_1, \dots, s_k$. Se reporta su promedio y su dispersión:

$$\bar{s} = \frac{1}{k}\sum_{j=1}^{k} s_j \qquad \text{y} \qquad \text{desv}(s_1, \dots, s_k)$$

- El **promedio** es una estimación más estable que un solo split: usa todas las filas de entrenamiento para validar (cada una, una vez).
- La **desviación** dice qué tanto depende el resultado de la partición. Sin ella, un número no dice nada sobre su incertidumbre.
- **Estratificada** (`StratifiedKFold`): cada fold conserva la proporción de "sí" (11,7 %), igual que `stratify=y` en la Sesión 3.

<details><summary><b>Para quien quiera más:</b> ¿por qué no usar la desviación como un intervalo de confianza?</summary>

Los $k$ scores no son independientes (los modelos comparten la mayor parte del entrenamiento), así que la desviación **subestima** la incertidumbre real. Sirve para comparar modelos de forma informal; para una prueba formal se usan validación cruzada repetida (`RepeatedStratifiedKFold`) o pruebas pareadas corregidas (Nadeau y Bengio, 2003).
</details>

![Dentro de cross_val_score](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S10/img/dentro_del_cross_val_score.png)

### ✏️ Completa — su primera validación cruzada
Use `cross_val_score` con el árbol de profundidad 5, `X_train`, `y_train`, el objeto `cv` ya creado y `scoring="roc_auc"`. Guarde el resultado en `scores_arbol`.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
arbol = DecisionTreeClassifier(max_depth=5, random_state=42)
scores_arbol = ...   # TODO: cross_val_score(modelo, X, y, cv=cv, scoring="roc_auc")
print("AUC por fold:", scores_arbol.round(3))
print(f"promedio {scores_arbol.mean():.3f} ± {scores_arbol.std():.3f}")

### La perilla de la validación cruzada: `k`

In [ ]:
fg.cv_k_efecto(arbol, X_train, y_train, ks=(2, 3, 5, 10, 20));

El promedio casi no se mueve después de `k = 5`; el tiempo sí crece. Por eso el estándar es **5 o 10**. Con datasets pequeños conviene 10 (cada ronda entrena con más filas).

## 3. Comparar modelos de verdad

Ahora los cuatro modelos del Módulo 2 con la **misma** validación cruzada. La logística y k-NN necesitan escalar; para eso usamos `make_pipeline`, que pega el escalador al modelo (la sección 5 explica por qué esto importa tanto).

### 🔍 Predice
¿Qué orden espera (de mejor a peor AUC)? ¿Alguna diferencia será "empate"?

**Tu predicción:** ____

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

modelos = {
    "árbol": arbol,
    "logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=25)),
}
resultados = {nombre: cross_val_score(m, X_train, y_train, cv=cv, scoring="roc_auc") for nombre, m in modelos.items()}

### ✏️ Completa — agregue el bosque
Agregue a `resultados` la validación cruzada de un `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1)`. Tarda unos 15 segundos (5 bosques de 200 árboles).

In [ ]:
bosque = ...                  # TODO: el RandomForestClassifier del enunciado
resultados["bosque"] = ...    # TODO: su validación cruzada (igual que los otros)
fg.cv_scores(resultados, titulo="Cuatro modelos, la misma validación cruzada (5 folds)");

## 4. 💥 El atajo de duplicar los "sí"

En la Sesión 5 vimos que con 11,7 % de "sí" los modelos casi nunca dicen "sí". Un truco muy popular en internet: **sobremuestrear**, es decir, duplicar filas de "sí" al azar hasta que haya tantas como "no". Hagámoslo y validemos con nuestra flamante validación cruzada.

In [ ]:
from sklearn.utils import resample

entrenamiento = X_train.assign(y=y_train.values)
si, no = entrenamiento[entrenamiento["y"] == 1], entrenamiento[entrenamiento["y"] == 0]
balanceado = pd.concat([no, resample(si, replace=True, n_samples=len(no), random_state=0)])
X_bal, y_bal = balanceado.drop(columns="y"), balanceado["y"]
print(f"antes: {len(entrenamiento):,} filas, {y_train.mean():.1%} 'sí'  →  después: {len(balanceado):,} filas, {y_bal.mean():.0%} 'sí'")

### 🔍 Predice
El bosque sin trucos tenía AUC ≈ 0,79 y recall ≈ 0,19 (Sesión 6). Con los datos balanceados así, ¿qué AUC y qué recall promete la validación cruzada?

**Tu predicción:** AUC ____ · recall ____

In [ ]:
from sklearn.model_selection import cross_validate

cv_dup = cross_validate(bosque, X_bal, y_bal, cv=cv, scoring=["roc_auc", "recall"])
print(f"Validación cruzada — AUC {cv_dup['test_roc_auc'].mean():.3f} · recall {cv_dup['test_recall'].mean():.3f}")

**¡AUC 0,96 y recall 0,90!** Un salto enorme. Antes de celebrar, hagamos lo que nunca hemos hecho en esta sesión: abrir la **prueba**, que el bosque no ha visto.

In [ ]:
from sklearn.metrics import recall_score

bosque_dup = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1).fit(X_bal, y_bal)
p = bosque_dup.predict_proba(X_test)[:, 1]
print(f"Prueba — AUC {roc_auc_score(y_test, p):.3f} · recall {recall_score(y_test, p >= 0.5):.3f}")

In [ ]:
fg.duplicados_fuga();

### La forma honesta
Si se quiere balancear, que sea **dentro** de cada ronda. La forma más simple ya la conocemos (Sesión 5): `class_weight="balanced"`, que pesa más los errores en "sí" sin copiar filas.

### ✏️ Completa — la versión honesta
Haga `cross_validate` del bosque con `class_weight="balanced"` sobre `X_train, y_train` (sin duplicar) con `scoring=["roc_auc", "recall"]`, y luego entrénelo y evalúelo en la prueba.

In [ ]:
bosque_bal = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced", random_state=42, n_jobs=-1)
cv_bal = ...   # TODO: cross_validate(bosque_bal, X_train, y_train, cv=cv, scoring=["roc_auc", "recall"])
p_bal = ...    # TODO: entrene bosque_bal con X_train, y_train y saque predict_proba(X_test)[:, 1]
fg.promesa_vs_realidad({
    "duplicar los 'sí'\nantes de validar": {"CV": {"AUC": cv_dup["test_roc_auc"].mean(), "recall": cv_dup["test_recall"].mean()},
                                            "prueba": {"AUC": roc_auc_score(y_test, p), "recall": recall_score(y_test, p >= 0.5)}},
    "class_weight='balanced'\n(dentro de cada ronda)": {"CV": {"AUC": cv_bal["test_roc_auc"].mean(), "recall": cv_bal["test_recall"].mean()},
                                                         "prueba": {"AUC": roc_auc_score(y_test, p_bal), "recall": recall_score(y_test, p_bal >= 0.5)}},
});

## 5. `Pipeline`: el preprocesamiento viaja con el modelo

Un `Pipeline` encadena pasos (escalar → modelo) y se comporta como **un solo modelo**: `.fit()` ajusta todos los pasos con los datos que recibe, en orden; `.predict()` transforma y predice. Dentro de `cross_val_score`, eso significa que el escalador se re-ajusta **en cada ronda** solo con el entrenamiento de esa ronda.

### ✏️ Completa — su pipeline
Construya `mi_pipeline` con `make_pipeline` (escalador estándar + k-NN con 25 vecinos) y evalúelo con `cross_validate` usando tres métricas: `["roc_auc", "recall", "f1"]`.

In [ ]:
mi_pipeline = ...   # TODO: make_pipeline(escalador, KNeighborsClassifier(n_neighbors=25))
res = ...           # TODO: cross_validate(mi_pipeline, X_train, y_train, cv=cv, scoring=[...])
pd.DataFrame({m: res[f"test_{m}"] for m in ["roc_auc", "recall", "f1"]}).agg(["mean", "std"]).round(3)

`cross_validate` (con *e*) devuelve varias métricas a la vez y los tiempos de entrenamiento. En la Sesión 11 este mismo pipeline entra a `GridSearchCV` para **buscar** el mejor `k` sin hacer trampa.

## 6. El examen final

![Tres conjuntos](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S10/img/tres_conjuntos.png)

El procedimiento completo, el que va en su proyecto:

1. Apartar la **prueba** al principio.
2. Con el resto, **validación cruzada** para elegir modelo (y en la Sesión 11, hiperparámetros).
3. Entrenar el elegido con **todo** el entrenamiento.
4. Evaluar **una sola vez** en la prueba y reportar ese número junto con el de la validación cruzada.

Si el número de prueba se parece al de la validación cruzada, la medición fue honesta. Si es mucho peor, algo se filtró.

In [ ]:
bosque.fit(X_train, y_train)
p_final = bosque.predict_proba(X_test)[:, 1]
print(f"bosque — validación cruzada: {resultados['bosque'].mean():.3f} ± {resultados['bosque'].std():.3f}")
print(f"bosque — prueba (una sola vez): {roc_auc_score(y_test, p_final):.3f}")

## ⚡ Mini-reto (opcional): la otra fuga famosa

Fabrique datos **sin ninguna relación** con la *y* y vea cómo una selección de variables hecha antes de validar "encuentra" un modelo excelente (ejemplo clásico de Hastie, Tibshirani y Friedman, *The Elements of Statistical Learning*, §7.10.2):

```python
rng = np.random.default_rng(0)
X_ruido = rng.normal(size=(200, 2000)); y_ruido = rng.integers(0, 2, 200)
from sklearn.feature_selection import SelectKBest
X_sel = SelectKBest(k=20).fit_transform(X_ruido, y_ruido)          # ✗ elige variables mirando TODA la y
cross_val_score(LogisticRegression(), X_sel, y_ruido, cv=5, scoring="roc_auc").mean()
cross_val_score(make_pipeline(SelectKBest(k=20), LogisticRegression()), X_ruido, y_ruido, cv=5, scoring="roc_auc").mean()   # ✓
```

¿Qué AUC da cada una? ¿Cuál es la verdad para datos de puro ruido?

In [ ]:
# Su análisis aquí

## 7. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 10) |
|---|---|
| Evaluación | un número **con su dispersión** (validación cruzada estratificada, k = 5 o 10); la prueba se abre una sola vez |
| Preprocesamiento | todo lo que aprende de los datos va **dentro** del `Pipeline` (escalar, imputar, balancear, seleccionar) |
| Modelo | comparar con la misma partición; si las diferencias caben en la dispersión, es empate |

**Las fugas del curso, todas:**

| Sesión | Qué se filtró | Cómo se ve |
|---|---|---|
| S3 | estadística calculada con todas las filas (tasa por dirección) | entrenamiento y prueba igual de buenos… y falsos |
| S4 | la *y* hecha de las *X* (puntajes por área) | R² = 1,000 |
| S6 | la consecuencia de la *y* (`duration`) | AUC 0,93 |
| S9 | el resumen junto a sus partes (IPM) | 75 municipios cambian de grupo |
| S10 | preprocesar antes de validar (duplicar "sí") | la validación promete 0,96, la prueba da 0,80 |

**Para la Sesión 11:** cada modelo tiene perillas (`max_depth`, `k`, `C`, `n_estimators`). ¿Cómo encontrar las mejores sin hacer trampa? `GridSearchCV` y `RandomizedSearchCV`, un vistazo a **PyCaret**, y la entrega **E4**.